## Composite

---

> **In one line.** A composite is a recursive tree type $T$ whose two shapes — a `Leaf` and a `Node` — present the *same interface*, so a single operation $f$, defined by a base case on leaves and a combining case on nodes, can be called identically on either, at any depth.

### 1. The recursive type

Everything starts from one self-referential definition. Let $T$ be the **tree type**: a value of type $T$ is *either* a terminal $\mathrm{Leaf}(v)$ that holds a single value $v$ directly and has no children (e.g. a single file), *or* a composite $\mathrm{Node}(T_1, \dots, T_n)$ that contains $n$ children, each of which is *itself* a tree $T_i$ (e.g. a folder of files and subfolders). Written as a grammar:

$$\boxed{\,T \;=\; \mathrm{Leaf}(v) \;\mid\; \mathrm{Node}(T_1, T_2, \dots, T_n)\,}$$

The recursion is the whole point: a $\mathrm{Node}$ is built from smaller $T$'s, so the same two shapes describe a structure of *arbitrary* depth.

### 2. One operation, two cases

We want an **operation** $f$ — say `size()` — that we can apply uniformly to any tree, leaf or node alike. Because $T$ has exactly two constructors, $f$ is fixed by exactly two cases. The **leaf case** $g(v)$ says how $f$ evaluates a terminal node with value $v$ (e.g. return the file's size directly). The **node case** combines the results of the children through a function $h$, applied to $f$ evaluated *recursively* on each child:

$$\boxed{\,f(T) = \begin{cases} g(v) & \text{if } T = \mathrm{Leaf}(v) \\[4pt] h\big(f(T_1), f(T_2), \dots, f(T_n)\big) & \text{if } T = \mathrm{Node}(\dots) \end{cases}\,}$$

This is a **tree homomorphism**: $f$ folds the structure of $T$ into a single result, with $g$ handling the atoms and $h$ stitching the branches. The recursion unfolds at a node and the answers fold back up from its children:

$$\mathrm{Node}(T_1,\dots,T_n) \;\xrightarrow{\,f\,}\; h\Big(\underbrace{f(T_1)}_{\text{recurse}}, \dots, \underbrace{f(T_n)}_{\text{recurse}}\Big) \;\xrightarrow{\,h\,}\; \text{one value}$$

### 3. The defining invariant

What makes this a *pattern* rather than just a recursive function is that the client never branches on the case. Both shapes implement the same shared **method contract** — write $\mathrm{interface}(\cdot)$ for the set of method signatures a shape exposes — and the two are required to be identical:

$$\boxed{\,\mathrm{interface}(\mathrm{Leaf}) = \mathrm{interface}(\mathrm{Node})\,} \qquad \text{(key invariant)}$$

So calling `size()` on a `Leaf` and on a `Node` looks the same at the call site; the case analysis lives *inside* $f$, never in the caller.

### 4. The three conditions

1. **Uniform interface** — both shapes satisfy the same contract, so the client treats them interchangeably:
   $$\mathrm{interface}(\mathrm{Leaf}) = \mathrm{interface}(\mathrm{Node}).$$
   The client calls the same method on a file as on a folder and never needs to check which it has.
2. **Recursive evaluation** — the node case recurses: each child applies $f$ independently and the results are combined by $h$,
   $$f(\mathrm{Node}(T_1,\dots,T_n)) = h\big(f(T_1), \dots, f(T_n)\big).$$
3. **Unlimited depth** — since a $\mathrm{Node}$ may itself contain $\mathrm{Node}$'s, the tree can be arbitrarily deep, yet $f$ handles every level by the *same* two cases.

&nbsp;

> 📁 A file system. A file has `size()` → $g(v)$. A folder has `size()` → $h(f(T_1), \dots, f(T_n))$ = sum of children. You call the same method on both. The recursion handles any depth.


### Exercise 9 — File System Tree

---

**Scenario:** Build `File` (leaf) and `Folder` (node) sharing the same interface. A `Folder`'s `size()` recursively sums all its children.

**Your task:** Implement both with `size()` and `display(indent)`. Verify that $f$ works identically whether called on a file or on a deeply nested folder.

```python
root = Folder("root", [
    File("a.txt", 100),
    Folder("sub", [File("b.txt", 200), File("c.txt", 50)]),
])
root.size()       # same method as File.size() -> 350
root.display()
```

**Hints**

- `File.size()` is $g(v)$ — return `self._size` directly. `Folder.size()` is $h(f(T_1), \dots)$ — return `sum(child.size() for child in self._children)`.
- Define a shared base class `Component` with abstract `size()` and `display()` — this declares the uniform-interface condition formally.

In [ ]:
from abc import ABC, abstractmethod

# --------------------------------
# Uniform interface — interface(Leaf) == interface(Node)

class Component(ABC):
    @abstractmethod
    def size(self): ...
    @abstractmethod
    def display(self, indent=0): ...

# --------------------------------
# Leaf: f(Leaf(v)) = g(v)

class File(Component):
    def __init__(self, name, size):
        self._name = name
        self._size = size

    def size(self):                          # g(v): return value directly
        ...

    def display(self, indent=0):
        ...                                  # e.g. print('  '*indent + f'{name} ({size})')

# --------------------------------
# Node: f(Node(T1..Tn)) = h(f(T1), ..., f(Tn))

class Folder(Component):
    def __init__(self, name, children=None):
        self._name = name
        self._children = children or []

    def size(self):                          # h(...): sum children recursively
        ...

    def display(self, indent=0):
        ...                                  # print self, then display each child at indent+1

# --------------------------------
root = Folder("root", [
    File("a.txt", 100),
    Folder("sub", [File("b.txt", 200), File("c.txt", 50)]),
])
print("total size:", root.size())            # expect 350
root.display()

### Exercise 10 — Organisation Chart

---

**Scenario:** An individual contributor (leaf) has a salary. A manager (node) has a salary *plus* subordinates. `get_salary_total()` must work identically on both.

**Your task:** Apply the same Composite structure — the leaf case $g(v)$ returns own salary; the node case $h$ sums own salary plus all subordinates' totals recursively.

```python
ceo = Manager("CEO", 500, [
    Employee("Dev A", 100),
    Manager("Lead", 300, [Employee("Dev B", 120), Employee("Dev C", 110)]),
])
ceo.get_salary_total()    # same method on leaf and node -> 1130
```

**Hints**

- This is structurally *identical* to the file system — replace `size()` with `get_salary_total()`. The pattern is the same; only the domain changes.
- `Employee.get_salary_total()` is $g(v)$ — return own salary. `Manager.get_salary_total()` is $h(\dots)$ — own salary `+ sum(sub.get_salary_total() for sub in subordinates)`.

In [ ]:
from abc import ABC, abstractmethod

# --------------------------------
# Uniform interface — same contract for IC and manager

class Staff(ABC):
    @abstractmethod
    def get_salary_total(self): ...

# --------------------------------
# Leaf: g(v) = own salary

class Employee(Staff):
    def __init__(self, name, salary):
        self._name = name
        self._salary = salary

    def get_salary_total(self):              # g(v)
        ...

# --------------------------------
# Node: h(...) = own salary + sum of subordinates

class Manager(Staff):
    def __init__(self, name, salary, subordinates=None):
        self._name = name
        self._salary = salary
        self._subordinates = subordinates or []

    def get_salary_total(self):              # h(f(T1), ..., f(Tn)) + own salary
        ...

# --------------------------------
ceo = Manager("CEO", 500, [
    Employee("Dev A", 100),
    Manager("Lead", 300, [Employee("Dev B", 120), Employee("Dev C", 110)]),
])
print("total payroll:", ceo.get_salary_total())   # expect 1130